# Setup Profile Learning for Existing Users

This notebook sets up profile learning for users that already exist in the triplestore.

**What This Does**:
1. Lists existing users in the triplestore
2. Allows selection of a user
3. Checks for existing trajectories
4. Adds initial preferences (if not already set)
5. Prepares user for profile learning

**Note**: Uses actual POI categories from the Trajecto system.

In [ ]:
import sys

import pandas as pd
from SPARQLWrapper import JSON, POST, SPARQLWrapper

# Import local helpers
sys.path.append('helpers')
from helpers.fuseki_config import query_endpoint, update_endpoint
from helpers.fuseki_queries import execute_sparql_query, format_results
from helpers.poi_categories import POI_CATEGORIES, get_all_categories, get_popular_categories
from helpers.profile_learning import LEARNING_RATE

In [ ]:
# Configure SPARQL endpoints
print(f"Query endpoint: {query_endpoint}")
print(f"Update endpoint: {update_endpoint}")

sparql_query = SPARQLWrapper(query_endpoint)
sparql_update = SPARQLWrapper(update_endpoint)
sparql_update.method = POST

## Step 1: List Existing Users

Fetch all users from the triplestore.

In [ ]:
def get_existing_users():
    """Fetch all users from the triplestore."""
    query = """
PREFIX user: <http://w3id.org/trajecto/user#>
PREFIX trajecto: <http://w3id.org/trajecto#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT DISTINCT ?user ?userId
WHERE {
    ?user trajecto:formsTrajectory ?trajectory .
    ?user rdfs:label ?userId
}
ORDER BY ?userId
"""

    result = execute_sparql_query(query_endpoint, query)
    users = format_results(result)

    return users


users = get_existing_users()
print(f"\nFound {len(users)} users in the triplestore:\n")

if users.empty:
        print("❌ No users found! Make sure trajectories are loaded with user:belongsToUser property.")
else:
    display(users)

## Step 2: Select User

Choose which user to setup for profile learning.

In [ ]:
# No longer need to manually select a user index
# The setup will loop through all users automatically in the cells below
print(f"Total users to process: {len(users)}")

##Step3:HelperFunctions

Definehelperfunctionsforcheckingtrajectoriesandprofilestatus.

In [ ]:
INTERACTION_SCORES = {
    'DominantInteraction': 1.0,
    'ImmersiveInteraction': 0.75,
    'MeaningfulInteraction': 0.5
}

CONFIDENCE_BASE_RATE = 0.15


def get_first_trajectory(user_uri):
    """Get the user's first trajectory with a meaningful interaction
    with a categorized POI.

    A first trajectory without one gives no preferences to seed, and a user
    without an initial snapshot is skipped by every Stage 9 update.
    """
    query = f"""
PREFIX poi: <http://w3id.org/trajecto/poi#>
PREFIX user: <http://w3id.org/trajecto/user#>
PREFIX trajecto: <http://w3id.org/trajecto#>
PREFIX datacron: <http://www.datacron-project.eu/datAcron#>
PREFIX dul: <http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#>

SELECT ?trajectory ?endTimestamp
WHERE {{
    <{user_uri}> trajecto:formsTrajectory ?trajectory .
    ?lastPos dul:isPartOf ?trajectory .
    ?lastPos trajecto:timestamp ?endTimestamp .
    FILTER NOT EXISTS {{
        ?lastPos trajecto:hasNextPosition ?anyNext .
    }}
    FILTER EXISTS {{
        ?pos dul:isPartOf ?trajectory ;
             trajecto:hasInteraction ?interaction .
        ?interaction a poi:MeaningfulInteraction ;
                     poi:interactsWith ?poi .
        ?poi a ?poiCategory .
        FILTER(STRSTARTS(STR(?poiCategory), "http://w3id.org/trajecto/poi#"))
        FILTER(?poiCategory != poi:SignificantPOI)
    }}
}}
ORDER BY ?endTimestamp
LIMIT 1
"""
    result = execute_sparql_query(query_endpoint, query)
    trajectories = format_results(result)

    if trajectories.empty:
        return None, None
    return trajectories.iloc[0]['trajectory'], trajectories.iloc[0]['endTimestamp']


def get_first_trajectory_preferences(trajectory_uri):
    """Derive initial preferences from the first trajectory's POI interactions.

    Uses interaction type as the preference score (Dominant=1.0, Immersive=0.75,
    Meaningful=0.5). When multiple interactions exist for the same category,
    keeps the strongest one.
    """
    query = f"""
PREFIX poi: <http://w3id.org/trajecto/poi#>
PREFIX trajecto: <http://w3id.org/trajecto#>
PREFIX dul: <http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#>

# trajecto:hasInteraction hangs off RawPosition, not Trajectory; the POI link
# is poi:interactsWith, not poi:atPOI; and a POI's category is an rdf:type
# subclass in the poi: namespace, not a poi:hasCategory value. With any of
# those wrong the query silently returns nothing and every user is skipped.
SELECT ?poiCategory ?interactionType
WHERE {{
    ?position dul:isPartOf <{trajectory_uri}> .
    ?position trajecto:hasInteraction ?interaction .
    ?interaction a ?interactionType ;
                 poi:interactsWith ?poi .
    ?poi a ?poiCategory .
    FILTER(STRSTARTS(STR(?poiCategory), "http://w3id.org/trajecto/poi#"))
    FILTER(?poiCategory != poi:SignificantPOI)
    FILTER(?interactionType IN (
        poi:DominantInteraction,
        poi:ImmersiveInteraction,
        poi:MeaningfulInteraction
    ))
}}
"""
    result = execute_sparql_query(query_endpoint, query)
    df = format_results(result)

    if df.empty:
        return []

    # Keep strongest interaction per category
    best = {}
    for _, row in df.iterrows():
        cat = str(row['poiCategory'])
        itype = str(row['interactionType']).split('#')[-1]
        score = INTERACTION_SCORES.get(itype, 0.5)
        if cat not in best or score > best[cat]:
            best[cat] = score

    return [
        {
            "category": cat,
            "score": round(score, 2),
            "confidence": round(score * CONFIDENCE_BASE_RATE, 2)
        }
        for cat, score in best.items()
    ]


def profile_exists(user_uri):
    """Check if a user profile already exists."""
    check_query = f"""
    PREFIX user: <http://w3id.org/trajecto/user#>

    ASK {{
        <{user_uri}> user:hasProfile ?profile .
    }}
    """
    sparql_query.setQuery(check_query)
    sparql_query.setReturnFormat(JSON)
    result = sparql_query.query().convert()
    return result['boolean']


def has_initial_preference(user_uri):
    """Check if user profile has initial preferences."""
    check_query = f"""
    PREFIX user: <http://w3id.org/trajecto/user#>

    ASK {{
        <{user_uri}> user:hasProfile ?profile .
        ?profile user:hasSnapshot ?snapshot.
        ?snapshot user:initialPreferences true.
    }}
    """
    sparql_query.setQuery(check_query)
    sparql_query.setReturnFormat(JSON)
    result = sparql_query.query().convert()
    return result['boolean']


# def generate_random_preferences():
#     """Generate random initial preferences."""
#     selected_categories = random.sample(POI_CATEGORIES, 5)
#     return [
#         {
#             "category": f"http://w3id.org/trajecto/poi#{cat}",
#             "score": round(random.uniform(0.5, 1.0), 2),
#             "confidence": round(random.uniform(0.1, 0.3), 2)
#         }
#         for cat in selected_categories
#     ]


def preferences_to_sparql_values(preferences):
    """Convert list of preference dicts to SPARQL VALUES format."""
    lines = []
    for pref in preferences:
        line = f'(<{pref["category"]}> {pref["score"]} {pref["confidence"]})'
        lines.append(line)
    return "\n".join(lines)


print("Helper functions defined.")


##Step4:AvailablePOICategories

ShowtheactualPOIcategoriesavailableintheTrajectosystem.

In [ ]:
# import random

print(f"\nAvailable POI Categories ({len(POI_CATEGORIES)} total):\n")

# Display in columns
categories_sorted = get_all_categories()
for i in range(0, len(categories_sorted), 3):
    row = categories_sorted[i:i+3]
    print(f"  {row[0]:<25} {row[1] if len(row) > 1 else '':<25} {row[2] if len(row) > 2 else ''}")

print("\n\nSuggested Popular Categories for Initial Preferences:")
for i, cat in enumerate(get_popular_categories(), 1):
    print(f"  {i:2d}. {cat}")


##Step5:SetupAllUsers

Loopthroughallusersandcreateprofileswithinitialpreferences.

In [ ]:
def setup_user_profile(user_uri, user_id):
    """Setup profile and initial preferences for a single user."""
    results = {
        'user_id': user_id,
        'user_uri': user_uri,
        'profile_created': False,
        'preferences_added': False,
        'skipped': False,
        'error': None
    }

    try:
        # Check if profile already has initial preferences
        if has_initial_preference(user_uri):
            results['skipped'] = True
            return results

        # Get first trajectory URI and timestamp
        first_traj_uri, first_timestamp = get_first_trajectory(user_uri)
        if not first_traj_uri:
            results['error'] = "No trajectories found"
            return results

        # Create profile if it doesn't exist
        if not profile_exists(user_uri):
            profile_uri = f"{user_uri}_profile"
            from datetime import datetime, timedelta
            profile_created_ts = (datetime.fromisoformat(first_timestamp) - timedelta(days=1)).isoformat()
            insert_profile = f"""
PREFIX user: <http://w3id.org/trajecto/user#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

INSERT DATA {{
    <{profile_uri}> a user:UserProfile ;
        user:learningRate {LEARNING_RATE} ;
        user:profileCreated "{profile_created_ts}"^^xsd:dateTime .
    <{user_uri}> user:hasProfile <{profile_uri}> .
}}
"""
            sparql_update.setQuery(insert_profile)
            sparql_update.query()
            results['profile_created'] = True

        # Derive initial preferences from first trajectory
        preferences = get_first_trajectory_preferences(first_traj_uri)
        if not preferences:
            results['error'] = "No POI interactions in first trajectory"
            return results

        snapshot_uri = f"{user_uri}_snapshot_0"

        # Stamped at the end of the trajectory it is derived from: an earlier
        # stamp would date the first trajectory's preferences before its start,
        # and make Stage 9 count that trajectory a second time.
        initial_snapshot_timestamp = first_timestamp

        initialize_query = f"""
PREFIX user: <http://w3id.org/trajecto/user#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

INSERT {{
    ?profile user:hasSnapshot <{snapshot_uri}> .

    <{snapshot_uri}>
        a user:ProfileSnapshot ;
        user:initialPreferences true ;
        user:updateTimestamp "{initial_snapshot_timestamp}"^^xsd:dateTime ;
        user:hasPreferences [
            a user:PoiCategoryPreference ;
            user:prefersPoiCategory ?category ;
            user:preferenceScore ?score ;
            user:confidenceLevel ?confidence
        ] .
}}
WHERE {{
    <{user_uri}> user:hasProfile ?profile .

    VALUES (?category ?score ?confidence) {{
    {preferences_to_sparql_values(preferences)}
    }}
}}
"""
        sparql_update.setQuery(initialize_query)
        sparql_update.query()
        results['preferences_added'] = True

    except Exception as e:
        results['error'] = str(e)

    return results


print("Setup function defined.")


##Step6:RunSetupforAllUsers

Executethesetupprocessforeachuserfoundinthetriplestore.

In [ ]:
# Process all users
setup_results = []

print(f"Processing {len(users)} users...\n")
print("-" * 80)

for idx, row in users.iterrows():
    user_uri = row['user']
    user_id = row['userId']

    print(f"[{idx + 1}/{len(users)}] Setting up user: {user_id}")

    result = setup_user_profile(user_uri, user_id)
    setup_results.append(result)

    if result['skipped']:
        print("    ⏭️  Skipped (already has initial preferences)")
    elif result['error']:
        print(f"    ❌ Error: {result['error']}")
    else:
        status = []
        if result['profile_created']:
            status.append("profile created")
        if result['preferences_added']:
            status.append("preferences added")
        print(f"    ✅ {', '.join(status)}")

print("-" * 80)
print("\nSetup complete!")

##Step7:Summary

Viewasummaryofthesetupresults.

In [ ]:
# Summary statistics
results_df = pd.DataFrame(setup_results)

profiles_created = results_df['profile_created'].sum()
preferences_added = results_df['preferences_added'].sum()
skipped = results_df['skipped'].sum()
errors = results_df['error'].notna().sum()

print("=" * 80)
print("SETUP SUMMARY")
print("=" * 80)
print(f"\nTotal users processed: {len(users)}")
print(f"  ✅ Profiles created:     {profiles_created}")
print(f"  ✅ Preferences added:    {preferences_added}")
print(f"  ⏭️  Skipped (existing):   {skipped}")
print(f"  ❌ Errors:               {errors}")

if errors > 0:
    print("\nUsers with errors:")
    error_users = results_df[results_df['error'].notna()][['user_id', 'error']]
    display(error_users)

print("\n" + "=" * 80)

##Step8:VerifyAllProfiles

Querythetriplestoretoconfirmallprofilesaresetupcorrectly.

In [ ]:
# Verify all user profiles
verify_all_query = """
PREFIX user: <http://w3id.org/trajecto/user#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?userId ?profile ?snapshot ?prefCount
WHERE {
    ?user rdfs:label ?userId .
    ?user user:hasProfile ?profile .
    ?profile user:hasSnapshot ?snapshot .
    ?snapshot user:initialPreferences true .

    {
        SELECT ?snapshot (COUNT(?pref) AS ?prefCount)
        WHERE {
            ?snapshot user:hasPreferences ?pref .
        }
        GROUP BY ?snapshot
    }
}
ORDER BY ?userId
"""

result = execute_sparql_query(query_endpoint, verify_all_query)
verification_df = format_results(result)

print("Users with initialized profiles:\n")
display(verification_df)

##(Optional)ViewPreferencesforaSpecificUser

Usethiscelltoinspectthepreferencesforanyindividualuser.

In [ ]:
# Change USER_INDEX to inspect a different user's preferences
USER_INDEX = 9  # Change this to view a different user

if not users.empty and len(users) > USER_INDEX:
    USER_URI = users.iloc[USER_INDEX]['user']
    USER_ID = users.iloc[USER_INDEX]['userId']

    verify_query = f"""
PREFIX user: <http://w3id.org/trajecto/user#>

SELECT ?category ?score ?confidence
WHERE {{
    <{USER_URI}> user:hasProfile ?profile .
    ?profile user:hasSnapshot ?snapshot .
    ?snapshot user:initialPreferences true ;
              user:hasPreferences ?pref .
    ?pref a user:PoiCategoryPreference ;
          user:prefersPoiCategory ?category ;
          user:preferenceScore ?score ;
          user:confidenceLevel ?confidence .
}}
"""

    sparql_query.setQuery(verify_query)
    sparql_query.setReturnFormat(JSON)
    results = sparql_query.query().convert()

    prefs_data = []
    for binding in results['results']['bindings']:
        if 'category' in binding:
            prefs_data.append({
                'Category': binding['category']['value'].split('#')[-1],
                'Score': float(binding['score']['value']),
                'Confidence': float(binding['confidence']['value'])
            })

    print(f"Preferences for user: {USER_ID}\n")
    if prefs_data:
        display(pd.DataFrame(prefs_data))
    else:
        print("No preferences found for this user.")
else:
    print("Invalid USER_INDEX or no users available.")

In [ ]:
# Validate KG state against stage-08-user-profiles SHACL shapes
from helpers.shacl_validator import validate_before_import, construct_validation_graph

proposed = construct_validation_graph('stage-08-user-profiles')
print(f"Fetched {len(proposed)} triples for validation")
report = validate_before_import(proposed, 'stage-08-user-profiles', halt_on_violations=True)